# {{TITLE}}

{{DATE}}, {{AUTHOR}}. An experiment folder made with `pixi run new-experiment`: the config beside this notebook, the results written into the folder by the last code cell.

## Purpose

_The question this experiment answers, in one or two sentences._

## Notes

_Decisions and surprises, added as they come._

In [ ]:
config_path = "config.yaml"  # the experiment config, relative to this folder
results_dir = "."  # where summary.json, metrics.csv and figures/ are written
checkpoint_dir = None  # a folder under RUNS from configs/local.yaml to keep the weights per epoch, none keeps none

## Config

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import display

import axis_of_travel_rnn as aot

started = datetime.now(timezone.utc).isoformat(timespec="seconds")
config = aot.load_config(config_path)
config

## Blocks: the blueprint's built folder, split by recording date

In [ ]:
loaded = aot.load_blocks(config.data)
print(f"{config.data.blueprint}, data id {loaded['data_id']}, {loaded['folder']}")
{name: len(blocks) for name, blocks in loaded["blocks"].items()}

## Windows: the training windows of each split

In [ ]:
windowed = aot.make_windows(loaded, config.windows)
{name: len(rows) for name, rows in windowed["windows"].items()}

## Dataset: the inputs scaled by the train split only

In [ ]:
datasets = aot.make_dataset(loaded, windowed, config.inputs)

## Train: early stopping on the validation split

In [ ]:
net, rows = aot.train(datasets, config.model, config.training, config.seed,
                      None if checkpoint_dir is None else Path(checkpoint_dir))

## Evaluate: the network against the persistence baseline on each split

In [ ]:
evaluation = aot.evaluate(net, datasets, config.training.device)
evaluation

## Measures: the hidden states of the test blocks

In [ ]:
found = aot.measures(net, datasets, config.analysis, config.seed, config.training.device)
{key: found[key] for key in ("fractions", "decoders") if key in found}

## Figures: each with its title, caption and data

In [ ]:
figures = aot.figures(config, loaded, rows, evaluation, found)
for fig in figures.values():
    display(fig)

## Save results: summary.json, metrics.csv and figures/ in the results folder

In [ ]:
finished = datetime.now(timezone.utc).isoformat(timespec="seconds")
summary = {"evaluation": evaluation, **{key: found[key] for key in ("fractions", "decoders") if key in found}}
aot.save_results(results_dir, config, loaded["data_id"], rows, summary, figures, started, finished)

## Outcome

_What the run showed, with the figures it rests on named by title. Then `pixi run export-experiment <this folder>/notebook.ipynb` writes notebook.html._